In [14]:

# -*- coding: utf-8 -*-

"""
CatBoost 信贷违约预测：特征工程 + 训练 + 预测 一体化脚本
-------------------------------------------------------------------
用法：
  1) 确保以下文件位于同一目录（默认 /mnt/data 或当前工作目录）：
     - 训练主数据: train.csv
     - 训练流水数据: train_bank_statement.csv
     - 测试主数据: testaa.csv
     - 测试流水数据: testaa_bank_statement.csv
  2) 执行：
     python catboost_pipeline.py
  3) 输出：
     - /mnt/data/models/ 目录下保存每折 CatBoost 模型
     - /mnt/data/submission.csv 预测结果（包含 id 和 default_prob）
     - /mnt/data/feature_importance.csv 特征重要性
     - /mnt/data/merge_debug_train.csv / merge_debug_test.csv 用于检查合并后的特征
说明：
  - 自动推断 ID 字段与目标字段
  - 自动识别日期字段并在流水侧做窗口聚合 (30/60/90/180/365 天)
  - 构造比率特征 (借/贷、最近/历史、均值/标准差等)
  - 防止数据泄漏：不使用目标编码，所有聚合在客户级完成
  - CatBoost 原生处理缺失与类别特征，无需额外编码
"""

import os, re, json, math, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

BASE = 'output_GPT'
np.random.seed(42)


# ===== Helpers: robust datetime & NaT sanitization =====
def _to_epoch_seconds(series):
    vals = pd.to_datetime(series, errors="coerce", utc=False)
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    mask = vals.notna()
    out.loc[mask] = vals.loc[mask].astype("int64") / 1e9  # seconds
    return out

def coerce_datetime_like_columns(df):
    """Convert any datetime64 or datetime-like object columns to epoch seconds (float)."""
    for c in df.columns:
        dt = str(df[c].dtype)
        if dt.startswith("datetime64"):
            df[c] = _to_epoch_seconds(df[c])
        elif df[c].dtype == "object":
            s = df[c]
            # If contains NaT object or 'NaT' string or obvious date-like strings, try convert
            try:
                has_nat_obj = s.map(lambda x: isinstance(x, type(pd.NaT))).any()
            except Exception:
                has_nat_obj = False
            has_nat_str = s.astype(str).str.upper().str.contains("NAT").any()
            looks_date = s.astype(str).str.contains(r"\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|\d{10}", regex=True).any()
            if has_nat_obj or has_nat_str or looks_date:
                converted = _to_epoch_seconds(s)
                if converted.notna().sum() > 0:
                    df[c] = converted
    return df

def sanitize_for_catboost(df, cat_cols):
    """Final guard: ensure no NaT/None in numeric features; convert 'NaT' strings to NaN; coerce numerics."""
    df = df.copy()
    # Convert datetime-like first
    df = coerce_datetime_like_columns(df)
    # Replace 'NaT' strings with NaN
    for c in df.columns:
        if df[c].dtype == "object":
            s = df[c].astype(str)
            if s.str.upper().eq("NAT").any():
                df.loc[s.str.upper().eq("NAT"), c] = np.nan
    # Try to coerce non-categorical object columns to numeric where meaningful
    for c in df.columns:
        if c not in cat_cols and df[c].dtype == "object":
            num = pd.to_numeric(df[c], errors="coerce")
            if num.notna().sum() > 0:
                df[c] = num
        # Replace Python None with np.nan
        df[c] = df[c].replace({None: np.nan, np.inf: np.nan, -np.inf: np.nan})
    return df
# ===== End Helpers =====


def load_csv_safe(path):
    if not os.path.exists(path):
        return None
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as e:
        for enc in ["utf-8", "gbk", "latin1"]:
            try:
                return pd.read_csv(path, low_memory=False, encoding=enc)
            except Exception:
                continue
        raise e

def infer_id_col(df):
    candidates = [c for c in df.columns if c.lower() in ["id","loan_id","user_id","customer_id","cust_id","account_id","client_id"]]
    if candidates:
        return candidates[0]
    for c in df.columns:
        if re.fullmatch(r".*id$", c, flags=re.I):
            return c
    return df.columns[0]

def infer_target_col(df):
    preferred = ["target","label","is_default","default","bad","y"]
    for name in preferred:
        for c in df.columns:
            if c.lower() == name:
                return c
    # binary 0/1
    for c in df.columns:
        vals = df[c].dropna().unique()
        if len(vals)==2 and set(vals) <= {0,1}:
            return c
    return None

def infer_date_col(df):
    date_like = [c for c in df.columns if re.search(r"(date|time|timestamp)", c, flags=re.I)]
    for c in date_like:
        ser = pd.to_datetime(df[c], errors="coerce", utc=False)
        if ser.notna().sum() > 0:
            return c
    return None

def coerce_str(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype(str)
    return df

def detect_amount_columns(df):
    """Heuristics to find amount/debit/credit/balance columns in statement data."""
    cols = df.columns.str.lower().tolist()
    amount = None
    balance = None
    debit = None
    credit = None
    typcol = None

    for cand in ["amount","amt","txn_amt","transaction_amount","money","value","sum"]:
        if cand in cols:
            amount = df.columns[cols.index(cand)]
            break
    for cand in ["balance","bal","acct_balance","running_balance"]:
        if cand in cols:
            balance = df.columns[cols.index(cand)]
            break
    for cand in ["debit","debit_amount","dr_amt","out_amount"]:
        if cand in cols:
            debit = df.columns[cols.index(cand)]
            break
    for cand in ["credit","credit_amount","cr_amt","in_amount"]:
        if cand in cols:
            credit = df.columns[cols.index(cand)]
            break
    for cand in ["type","txn_type","transaction_type","debit_credit"]:
        if cand in cols:
            typcol = df.columns[cols.index(cand)]
            break
    return amount, balance, debit, credit, typcol

def build_statement_features(stmt, id_col, date_col):
    """
    Aggregate transaction-level data to customer-level features.
    Returns a DataFrame keyed by id_col.
    """
    if stmt is None or stmt.empty:
        return pd.DataFrame(columns=[id_col])

    stmt = stmt.copy()
    stmt[date_col] = pd.to_datetime(stmt[date_col], errors="coerce")
    stmt = stmt[stmt[date_col].notna()]
    amount, balance, debit, credit, typcol = detect_amount_columns(stmt)

    # If only amount is present, try to infer sign by type if available, else keep as-is
    if amount is not None:
        stmt[amount] = pd.to_numeric(stmt[amount], errors="coerce")
    if balance is not None:
        stmt[balance] = pd.to_numeric(stmt[balance], errors="coerce")
    if debit is not None:
        stmt[debit] = pd.to_numeric(stmt[debit], errors="coerce")
    if credit is not None:
        stmt[credit] = pd.to_numeric(stmt[credit], errors="coerce")

    # Signed amount synthesis
    signed_amount = None
    if amount is not None:
        signed_amount = amount
        # If typcol exists with debit/credit tags, map to signs conservatively
        if typcol is not None:
            typ = stmt[typcol].astype(str).str.lower()
            sign = np.where(typ.str.contains("debit|支出|转出|付款"), -1, np.where(typ.str.contains("credit|收入|转入|收款"), 1, 1))
            stmt["_signed_amount"] = stmt[amount] * sign
            signed_amount = "_signed_amount"
    elif debit is not None or credit is not None:
        stmt["_signed_amount"] = stmt.get(credit, 0).fillna(0) - stmt.get(debit, 0).fillna(0)
        signed_amount = "_signed_amount"

    # counterparties unique count (optional)
    cp_cols = [c for c in stmt.columns if re.search(r"(counterparty|merchant|opposite|name|account_no)", c, flags=re.I)]
    cp_cols = [c for c in cp_cols if c not in [id_col, date_col, amount, balance, debit, credit, typcol]]

    # Base aggregations
    agg = stmt.groupby(id_col).agg(
        txn_count = (date_col, "count"),
        first_txn = (date_col, "min"),
        last_txn  = (date_col, "max"),
    )
    if signed_amount is not None:
        agg2 = stmt.groupby(id_col)[signed_amount].agg(["sum","mean","std","min","max","median"]).add_prefix("amt_")
        agg = agg.join(agg2, how="left")
    if debit is not None:
        agg3 = stmt.groupby(id_col)[debit].agg(["sum","mean","std","min","max","median"]).add_prefix("debit_")
        agg = agg.join(agg3, how="left")
    if credit is not None:
        agg4 = stmt.groupby(id_col)[credit].agg(["sum","mean","std","min","max","median"]).add_prefix("credit_")
        agg = agg.join(agg4, how="left")
    if balance is not None:
        agg5 = stmt.groupby(id_col)[balance].agg(["mean","std","min","max","median"]).add_prefix("bal_")
        agg = agg.join(agg5, how="left")
    if cp_cols:
        for c in cp_cols[:3]:  # limit to 3 columns to control feature blowup
            agg[f"nunique_{c}"] = stmt.groupby(id_col)[c].nunique(dropna=True)

    # Active days & intensity
    agg["active_days"] = (agg["last_txn"] - agg["first_txn"]).dt.days.clip(lower=0)
    agg["txn_intensity_per_day"] = agg["txn_count"] / (agg["active_days"] + 1)

    # Windowed aggregates
    max_date = stmt[date_col].max()
    for w in [30, 60, 90, 180, 365]:
        mask = stmt[date_col] >= (max_date - pd.Timedelta(days=w))
        sub = stmt.loc[mask]
        grp = sub.groupby(id_col).agg(last_txn_count=(date_col, "count"))
        agg[f"txn_count_{w}d"] = grp["last_txn_count"]
        if signed_amount is not None:
            grp2 = sub.groupby(id_col)[signed_amount].agg(["sum","mean","std","median"]).add_prefix(f"amt_{w}d_")
            for col in grp2.columns:
                agg[col] = grp2[col]

    # Ratio features
    def safe_ratio(a, b):
        return a / (b.replace(0, np.nan))  # keep NaN; CatBoost can handle

    if "credit_sum" in agg.columns and "debit_sum" in agg.columns:
        agg["ratio_amt_credit_to_debit"] = safe_ratio(agg["credit_sum"], agg["debit_sum"])
    if "amt_sum" in agg.columns and "amt_std" in agg.columns:
        agg["ratio_amt_mean_to_std"] = safe_ratio(agg["amt_mean"], agg["amt_std"])
    if "txn_count_30d" in agg.columns:
        agg["ratio_txn_30d_to_all"] = safe_ratio(agg["txn_count_30d"], agg["txn_count"])
    if "amt_30d_sum" in agg.columns and "amt_sum" in agg.columns:
        agg["ratio_amt_30d_to_all"] = safe_ratio(agg["amt_30d_sum"], agg["amt_sum"])

    agg.reset_index(inplace=True)
    return agg

def basic_row_ratios(df, exclude_cols):
    """
    Construct a limited set of within-row ratios to avoid combinatorial explosion.
    For numeric columns (<= 20), build pairwise X / (Y+eps) for top-variance columns.
    """
    num_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) == 0:
        return df, []
    # pick up to 12 numeric columns with highest variance
    var = df[num_cols].var().sort_values(ascending=False)
    picked = var.index.tolist()[:12]
    new_cols = []
    for i in range(len(picked)):
        for j in range(i+1, len(picked)):
            a, b = picked[i], picked[j]
            col = f"ratio__{a}__over__{b}"
            df[col] = df[a] / (df[b].replace(0, np.nan))
            new_cols.append(col)
    return df, new_cols

def prepare_dataset(main_df, stmt_df, is_train=True, id_col=None, tgt_col=None):
    if id_col is None:
        id_col = infer_id_col(main_df)
    main_df = main_df.copy()
    main_df[id_col] = main_df[id_col].astype(str)

    # infer target in train
    if is_train and tgt_col is None:
        tgt_col = infer_target_col(main_df)

    # drop target leakage columns if any (postfix like *_target or explicitly 'target_leak')
    leak_cols = [c for c in main_df.columns if re.search(r"(leak|target_leak)", c, flags=re.I)]
    if leak_cols:
        main_df = main_df.drop(columns=leak_cols)

    # Merge statement features (left join)
    if stmt_df is not None and not stmt_df.empty:
        stmt_id = infer_id_col(stmt_df)
        date_col = infer_date_col(stmt_df)
        stmt_df = stmt_df.copy()
        stmt_df[stmt_id] = stmt_df[stmt_id].astype(str)
        if date_col is not None:
            stmt_feat = build_statement_features(stmt_df, id_col=stmt_id, date_col=date_col)
        else:
            # no date: aggregate simple statistics only
            stmt_feat = stmt_df.groupby(stmt_id).size().reset_index(name="txn_count")
        # rename id col to match main id
        if stmt_feat.columns[0] != id_col:
            stmt_feat = stmt_feat.rename(columns={stmt_feat.columns[0]: id_col})
        feat_df = main_df.merge(stmt_feat, on=id_col, how="left", validate="m:1")
    else:
        feat_df = main_df.copy()

    # Identify categorical columns (object, string-like) excluding id and target
    cat_cols = [c for c in feat_df.columns if c not in [id_col, tgt_col] and feat_df[c].dtype == "object"]
    # Cast categorical to string to avoid unexpected numeric parsing
    for c in cat_cols + [id_col]:
        if c in feat_df.columns:
            feat_df[c] = feat_df[c].astype(str)

    # Build basic row-level ratio features using numeric columns only, excluding id/target
    exclude = [id_col]
    if tgt_col is not None:
        exclude.append(tgt_col)
    feat_df, ratio_cols = basic_row_ratios(feat_df, exclude_cols=exclude)

    # Missing value strategy:
    # - Keep NaN for CatBoost (it can handle NaN)
    # - For count-like derived features from statements, fillna(0)
    count_like = [c for c in feat_df.columns if re.search(r"(count|nunique|_30d|_60d|_90d|_180d|_365d)$", c)]
    for c in count_like:
        if c in feat_df.columns:
            feat_df[c] = feat_df[c].fillna(0)

    # Return prepared dataset, cat columns list, and recorded metadata
    meta = {
        "id_col": id_col,
        "tgt_col": tgt_col,
        "cat_cols": cat_cols,
        "ratio_cols": ratio_cols,
    }
    # Convert datetime-like columns now (train/test consistent)
    feat_df = coerce_datetime_like_columns(feat_df)
    return feat_df, meta

    # --- Convert datetime-like columns to numeric seconds since epoch (NaN preserved) ---
    for _col in feat_df.columns:
        if str(feat_df[_col].dtype).startswith("datetime64"):
            _vals = pd.to_datetime(feat_df[_col], errors="coerce", utc=False)
            _out = pd.Series(np.nan, index=feat_df.index, dtype="float64")
            _mask = _vals.notna()
            _out.loc[_mask] = _vals.loc[_mask].astype("int64") / 1e9  # seconds
            feat_df[_col] = _out

def get_cat_indices(df, cat_cols):
    # CatBoost expects indices; ensure valid and present
    idxs = [df.columns.get_loc(c) for c in cat_cols if c in df.columns]
    return idxs


In [15]:

train_main = load_csv_safe('train/train.csv')
train_stmt = load_csv_safe('train/train_bank_statement.csv')
test_main  = load_csv_safe('testaa/testaa.csv')
test_stmt  = load_csv_safe('testaa/testaa_bank_statement.csv')

assert train_main is not None, "训练主数据 train.csv 未找到"
# Prepare train
train_df, meta = prepare_dataset(train_main, train_stmt, is_train=True)
id_col = meta["id_col"]
tgt_col = meta["tgt_col"]
cat_cols = meta["cat_cols"]

assert tgt_col is not None, "无法在训练集中推断目标列(如 target/label/is_default/default/bad/y)，请手动修改脚本。"

# Separate X/y
y = train_df[tgt_col].astype(int)
X = train_df.drop(columns=[tgt_col])

# Final sanitation to avoid NaT/None issues
X = sanitize_for_catboost(X, cat_cols)

cat_indices = get_cat_indices(X, cat_cols)

# Class balance -> class_weights
pos_rate = y.mean()
class_weights = None
if pos_rate > 0 and pos_rate < 1:
    # weight inversely proportional to frequency
    w1 = 0.5 / pos_rate
    w0 = 0.5 / (1 - pos_rate)
    class_weights = [w0, w1]

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2025)
oof = np.zeros(len(X))
models = []
feature_importances = []

os.makedirs(os.path.join(BASE, "models"), exist_ok=True)

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    X_tr = sanitize_for_catboost(X_tr, cat_cols)
    X_val = sanitize_for_catboost(X_val, cat_cols)
    train_pool = Pool(X_tr, y_tr, cat_features=cat_indices)
    valid_pool = Pool(X_val, y_val, cat_features=cat_indices)

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=4.0,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=2025,
        od_type="Iter",
        od_wait=100,
        verbose=200,
        class_weights=class_weights,
        task_type='GPU'
    )
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof[val_idx] = model.predict_proba(valid_pool)[:,1]
    models.append(model)

    # Save per-fold model
    model_path = os.path.join(BASE, "models", f"catboost_fold{fold}.cbm")
    model.save_model(model_path)

    # Feature importance
    fi = pd.DataFrame({
        "feature": X.columns,
        "importance": model.get_feature_importance(train_pool, type="FeatureImportance")
    })
    fi["fold"] = fold
    feature_importances.append(fi)

cv_auc = roc_auc_score(y, oof)
print(f"[CV] OOF AUC = {cv_auc:.6f}")
pd.DataFrame({"id": X[id_col], "y_true": y, "oof_pred": oof}).to_csv(os.path.join(BASE, "oof_pred.csv"), index=False)

# Aggregate feature importances
fi_all = pd.concat(feature_importances, axis=0)
fi_mean = fi_all.groupby("feature", as_index=False)["importance"].mean().sort_values("importance", ascending=False)
fi_mean.to_csv(os.path.join(BASE, "feature_importance.csv"), index=False)

# Save merged training for debugging
X.to_csv(os.path.join(BASE, "merge_debug_train.csv"), index=False)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6339207	best: 0.6339207 (0)	total: 38.6ms	remaining: 1m 17s
200:	test: 0.6682355	best: 0.6682355 (200)	total: 7.61s	remaining: 1m 8s
bestTest = 0.6690064669
bestIteration = 254
Shrink model to first 255 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6379919	best: 0.6379919 (0)	total: 37.9ms	remaining: 1m 15s
200:	test: 0.6698177	best: 0.6698177 (200)	total: 7.26s	remaining: 1m 4s
400:	test: 0.6707330	best: 0.6711427 (391)	total: 13.9s	remaining: 55.3s
600:	test: 0.6711630	best: 0.6715546 (544)	total: 20.9s	remaining: 48.7s
bestTest = 0.6715545654
bestIteration = 544
Shrink model to first 545 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6304998	best: 0.6304998 (0)	total: 33.1ms	remaining: 1m 6s
200:	test: 0.6677189	best: 0.6679575 (181)	total: 7.05s	remaining: 1m 3s
400:	test: 0.6701528	best: 0.6703160 (393)	total: 14.4s	remaining: 57.4s
bestTest = 0.6705832779
bestIteration = 436
Shrink model to first 437 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6406908	best: 0.6406908 (0)	total: 36.1ms	remaining: 1m 12s
200:	test: 0.6714959	best: 0.6715443 (198)	total: 7.99s	remaining: 1m 11s
400:	test: 0.6740046	best: 0.6741436 (393)	total: 21s	remaining: 1m 23s
bestTest = 0.6742776632
bestIteration = 449
Shrink model to first 450 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6354131	best: 0.6354131 (0)	total: 77.8ms	remaining: 2m 35s
200:	test: 0.6603214	best: 0.6604074 (197)	total: 13.1s	remaining: 1m 57s
400:	test: 0.6619165	best: 0.6620089 (300)	total: 26s	remaining: 1m 43s
bestTest = 0.6620089412
bestIteration = 300
Shrink model to first 301 iterations.
[CV] OOF AUC = 0.669373


In [16]:
# Predict on test if available
if test_main is not None:
    test_df, meta_test = prepare_dataset(test_main, test_stmt, is_train=False, id_col=id_col, tgt_col=None)
    # Align columns (add missing columns in test to match training)
    missing_cols = [c for c in X.columns if c not in test_df.columns]
    for c in missing_cols:
        test_df[c] = np.nan
    # Keep same column order
    test_df = test_df[X.columns]
    test_df = sanitize_for_catboost(test_df, cat_cols)

    test_pool = Pool(test_df, cat_features=get_cat_indices(test_df, cat_cols))
    preds = np.zeros(len(test_df))
    for model in models:
        preds += model.predict_proba(test_pool)[:,1]
    preds /= len(models)

    sub = pd.DataFrame({id_col: test_df[id_col], "default_prob": preds})
    sub.to_csv(os.path.join(BASE, "submission.csv"), index=False)
    test_df.to_csv(os.path.join(BASE, "merge_debug_test.csv"), index=False)
    print("[DONE] 预测文件已输出：/mnt/data/submission.csv")
else:
    print("未发现测试集(testaa.csv)，已完成训练与特征重要性输出。")

[DONE] 预测文件已输出：/mnt/data/submission.csv
